# Predicting Road Traffic Accident Severity Using Machine Learning

## 1. Introduction

Road traffic accidents have serious social and economic effects, including injuries, loss of life, property damage, and pressure on emergency services. This project aims to use machine learning to study road traffic accident data, identify factors related to accident severity, and develop models that can predict the severity of an accident.

## 1.1 Dataset Overview

The dataset used in this project was collected from Addis Ababa sub-city police departments. It contains 12,316 accident records and 32 features covering the period from 2017 to 2020. The data includes information about drivers, vehicles, road conditions, weather conditions, casualties, and accident severity.

## 1.2 Data Preparation

This section covers the steps taken to inspect and clean the dataset before it is used for analysis and machine learning. The process includes checking for missing values, identifying duplicate records, standardising text entries, and checking that the data is ready to be shared with the rest of the project team.


# 2. Data Collection

The dataset is provided as a CSV file inside a ZIP archive. The next cells import the required tools and load the data.

### 2.1 Import the Required Libraries

In [165]:
import pandas as pd
import zipfile

### 2.2 Load the Dataset

In [167]:
with zipfile.ZipFile("RTA Dataset.csv.zip", "r") as archive:
    with archive.open("RTA Dataset.csv") as file:
        df = pd.read_csv(file)

### 2.3 Create a Working Copy

In [169]:
df_clean = df.copy()

### 2.4 Check the Original Dataset Size

In [171]:
print("Original dataset shape:", df.shape)

Original dataset shape: (12316, 32)


# 3. Data Cleaning and Preprocessing

This section checks the original data, reviews missing values, cleans text entries, and checks the cleaned dataset before saving it.

## 3.1 Check the Dataset Size

In [174]:
print("Dataset shape:", df_clean.shape)

Dataset shape: (12316, 32)


## 3.2 Review the Column Names

In [176]:
print(df_clean.columns.tolist())

['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver', 'Educational_level', 'Vehicle_driver_relation', 'Driving_experience', 'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle', 'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians', 'Road_allignment', 'Types_of_Junction', 'Road_surface_type', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision', 'Number_of_vehicles_involved', 'Number_of_casualties', 'Vehicle_movement', 'Casualty_class', 'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity', 'Work_of_casuality', 'Fitness_of_casuality', 'Pedestrian_movement', 'Cause_of_accident', 'Accident_severity']


## 3.3 Check Column Data Types and Non-Missing Counts

In [178]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12316 entries, 0 to 12315
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Time                         12316 non-null  object
 1   Day_of_week                  12316 non-null  object
 2   Age_band_of_driver           12316 non-null  object
 3   Sex_of_driver                12316 non-null  object
 4   Educational_level            11575 non-null  object
 5   Vehicle_driver_relation      11737 non-null  object
 6   Driving_experience           11487 non-null  object
 7   Type_of_vehicle              11366 non-null  object
 8   Owner_of_vehicle             11834 non-null  object
 9   Service_year_of_vehicle      8388 non-null   object
 10  Defect_of_vehicle            7889 non-null   object
 11  Area_accident_occured        12077 non-null  object
 12  Lanes_or_Medians             11931 non-null  object
 13  Road_allignment              12

## 3.4 Count Actual Missing Values in Each Column

In [180]:
missing_values = df_clean.isnull().sum()
print(missing_values[missing_values > 0])

Educational_level           741
Vehicle_driver_relation     579
Driving_experience          829
Type_of_vehicle             950
Owner_of_vehicle            482
Service_year_of_vehicle    3928
Defect_of_vehicle          4427
Area_accident_occured       239
Lanes_or_Medians            385
Road_allignment             142
Types_of_Junction           887
Road_surface_type           172
Type_of_collision           155
Vehicle_movement            308
Work_of_casuality          3198
Fitness_of_casuality       2635
dtype: int64


## 3.5 Check for Duplicate Rows

In [182]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 0


## 3.6 Summarise Missing Values in the Original Dataset

This summary uses `df`, the original dataset, so it shows actual missing values before cleaning. Text entries such as `unknown` and `na` are not counted as missing by `isnull()`.

In [184]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df)) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
Defect_of_vehicle,4427,35.945112
Service_year_of_vehicle,3928,31.893472
Work_of_casuality,3198,25.966223
Fitness_of_casuality,2635,21.394933
Type_of_vehicle,950,7.713543
Types_of_Junction,887,7.202014
Driving_experience,829,6.731082
Educational_level,741,6.016564
Vehicle_driver_relation,579,4.701202
Owner_of_vehicle,482,3.913608


## 3.7 Remove Unnecessary Spaces from Text Entries

This step removes spaces at the beginning or end of text values. It does not change the meaning of the values.

In [186]:
text_columns = df_clean.select_dtypes(include="object").columns

for column in text_columns:
    df_clean[column] = df_clean[column].apply(
        lambda value: value.strip() if isinstance(value, str) else value
    )

## 3.8 Standardise Entries Labelled `unknown`

This changes text entries written as `unknown` to `Unknown`. It does not combine them with actual blank values.

In [188]:
for column in text_columns:
    df_clean[column] = df_clean[column].apply(
        lambda value: "Unknown"
        if isinstance(value, str) and value.lower() == "unknown"
        else value
    )

## 3.9 Handle `na` in Casualty-Related Columns

In these four columns, entries written as `na` are changed to `Not recorded`. This is a cleaning rule used in this notebook; the original data documentation should be checked to confirm the meaning of `na`.

In [190]:
casualty_columns = [
    "Casualty_class",
    "Sex_of_casualty",
    "Age_band_of_casualty",
    "Casualty_severity"
]

for column in casualty_columns:
    df_clean[column] = df_clean[column].apply(
        lambda value: "Not recorded"
        if isinstance(value, str) and value.lower() == "na"
        else value
    )

## 3.10 Label Actual Missing Text Values Separately

Actual blank values (`NaN`) are labelled `Missing`. This keeps them separate from entries explicitly written as `Unknown`.

In [192]:
for column in text_columns:
    df_clean[column] = df_clean[column].fillna("Missing")

## 3.11 Validate the Cleaned Dataset Size

Check that the number of rows and columns has not changed during cleaning.

In [194]:
print("Dataset shape:", df_clean.shape)

Dataset shape: (12316, 32)


## 3.12 Check for Duplicate Rows After Cleaning

In [196]:
print("Duplicate rows:", df_clean.duplicated().sum())

Duplicate rows: 0


## 3.13 Check Remaining Actual Missing Values

In [198]:
print("Remaining missing values:", df_clean.isnull().sum().sum())

Remaining missing values: 0


## 3.14 Check the Target Variable

In [200]:
print(df_clean["Accident_severity"].value_counts(dropna=False))

Slight Injury     10415
Serious Injury     1743
Fatal injury        158
Name: Accident_severity, dtype: int64


## 3.15 Check Missing Values in the Target

In [202]:
print("Missing target values:", df_clean["Accident_severity"].isnull().sum())

Missing target values: 0


# 4. Export the Cleaned Dataset
The final cell saves the cleaned data as a CSV file without adding an extra index column. Share this file with the team for the next stages of the project.


In [204]:
df_clean.to_csv("RTA_Dataset_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
